# 12 · ARIMA

Task 16.6. One ARMA(p, q) model per stock on daily log returns, order chosen by AIC on the first fold
(p, q ≤ 2), parameters re-estimated monthly, and extended with each new return in between. The h-day forecast is the
sum of the next h daily forecasts; `p_up` comes from its forecast distribution.

ARIMA forecasts close-to-close returns, so its forecast starts at the close of d−1 while the labels start at the open of d.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels, majority_guess, with_regimes
from src.models.data import label_frame, load_panel, regimes
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
RETRAIN = False          # True retrains even when saved forecasts match the current settings
panel = load_panel(cfg)
labels = label_frame(panel)
fc = pd.concat([run_or_load(m, cfg, "validation", panel=panel, retrain=RETRAIN) for m in ["arima", "base_rate"]],
               ignore_index=True)
metrics = evaluate(fc, labels)
metrics[metrics.model == "arima"].to_csv(figures_dir(cfg) / "arima_validation.csv", index=False)
metrics[["model", "horizon_days", "n", "accuracy", "balanced_accuracy", "f1_up", "brier", "log_loss", "predicted_up",
         "vol_mae"]]

## Orders chosen per stock

In [ ]:
import json
folds = sorted((cfg.models_dir / "arima" / "validation").glob("fold_*"))
first = json.loads((folds[0] / "params.json").read_text())
orders = pd.Series({s: f"ARMA({v['order'][0]},{v['order'][1]})" for s, v in first.items()})
print(f"{len(folds)} monthly folds; {len(first)} stocks")
orders.value_counts().rename("stocks").to_frame()

## Accuracy and probability quality against the base rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
dot_compare(axes[0], metrics, "accuracy", ["arima", "base_rate"], panel.horizons, reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy")
dot_compare(axes[1], metrics, "brier", ["arima", "base_rate"], panel.horizons, fmt="{:.4f}")
axes[1].set_title("Brier score (lower is better)")
fig.suptitle("ARIMA vs base rate, validation 2023", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, cfg, "12_arima_vs_base_rate")
plt.show()

In [ ]:
a5 = fc[(fc.model == "arima") & (fc.horizon_days == 5)]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(a5.p_up, bins=50, color=MODEL_COLORS["arima"], **bar_style())
ax.axvline(0.5, color=TEXT_2, linewidth=1, linestyle="--")
ax.set_xlabel("forecast probability of a 5-day rise")
ax.set_ylabel("stock-days")
ax.set_title("ARIMA's 5-day probabilities stay close to 50%")
ax.grid(axis="x", visible=False)
save(fig, cfg, "12_arima_p_up_hist")
plt.show()
a5.p_up.describe(percentiles=[0.05, 0.5, 0.95]).to_frame("p_up (5-day)")

## Findings

1. **Direction:** ARIMA is below the base rate at every horizon: 47.8% / 55.4% / 58.8% vs 53.0% / 56.2% / 60.6%, with balanced accuracy 49.9% / 50.0% / 49.3% (no skill). Its Brier score is marginally better only at 5 days (0.2483 vs 0.2485).
2. **Orders:** 23 of the 49 stocks get ARMA(0,0) (the forecast is the mean return): daily returns carry little linear autocorrelation.
3. **Probabilities stay near 50%:** the 5-day p_up has median 0.525 and 90% of values between 0.495 and 0.553, so ARIMA predicts 'up' 93% of the time at 5 days.
4. **Volatility:** ARIMA's constant variance gives MAE 0.0130 / 0.0096 / 0.0086, worse than the 20-day historical volatility at every horizon.
5. **Coverage:** every stock-day of 2023 has a forecast. SHRIRAMFIN, which lacked history before Dec 2022, is now covered from its SRTRANSFIN prices.